In [ ]:
import pandas as pd

In [ ]:
datiosx=pd.read_json(f"https://www.datos.gov.co/resource/jbjy-vk9h.json?$limit=10000")

In [ ]:
datiosx["proceso_de_compra"]

,proceso_de_compra
0,CO1.BDOS.4149871
1,CO1.BDOS.9186971
2,CO1.BDOS.3565674
3,CO1.BDOS.2845170
4,CO1.BDOS.10204764
...,...
9995,CO1.BDOS.8480447
9996,CO1.BDOS.483821
9997,CO1.BDOS.3744819
9998,CO1.BDOS.2647430


In [ ]:
datiosdocumentos = pd.read_json("https://www.datos.gov.co/resource/dmgg-8hin.json?proceso=CO1.BDOS.7691828")

In [ ]:
datiosdocumentos

,n_mero_de_contrato,id_documento,proceso,nombre_archivo,tamanno_archivo,extensi_n,descripci_n,fecha_carga,entidad,nit_entidad,url_descarga_documento
0,CO1.PCCNTR.7552865,606117172,CO1.BDOS.7691828,CEDULA REVISOR FISCAL.pdf,336600,pdf,CEDULA REVISOR FISCAL.pdf,2025-04-28T00:00:00.000,INSTITUTO DEPARTAMENTAL DE DEPORTES DE ANTIOQUIA,811007127,{'url': 'https://community.secop.gov.co/Public...
1,CO1.PCCNTR.7552865,768779506,CO1.BDOS.7691828,07 Informe de Supervision y Recibo a Satisfacc...,344972,pdf,07 Informe de Supervision y Recibo a Satisfacc...,2026-03-23T00:00:00.000,INSTITUTO DEPARTAMENTAL DE DEPORTES DE ANTIOQUIA,811007127,{'url': 'https://community.secop.gov.co/Public...
2,CO1.PCCNTR.7552865,705073627,CO1.BDOS.7691828,05 INFORME DE SUPERVICION Y RECIBO A SATISFACC...,300462,pdf,05 INFORME DE SUPERVICION Y RECIBO A SATISFACC...,2025-11-25T00:00:00.000,INSTITUTO DEPARTAMENTAL DE DEPORTES DE ANTIOQUIA,811007127,{'url': 'https://community.secop.gov.co/Public...
3,CO1.PCCNTR.7552865,705073633,CO1.BDOS.7691828,ad0800167494000250000027d.zip,73781,zip,ad0800167494000250000027d.zip,2025-11-25T00:00:00.000,INSTITUTO DEPARTAMENTAL DE DEPORTES DE ANTIOQUIA,811007127,{'url': 'https://community.secop.gov.co/Public...
4,CO1.PCCNTR.7552865,641460924,CO1.BDOS.7691828,INFORME DE SUPERVISION Y RECIBO A SATISFACCION...,283724,pdf,INFORME DE SUPERVISION Y RECIBO A SATISFACCION...,2025-07-23T00:00:00.000,INSTITUTO DEPARTAMENTAL DE DEPORTES DE ANTIOQUIA,811007127,{'url': 'https://community.secop.gov.co/Public...
...,...,...,...,...,...,...,...,...,...,...,...
247,CO1.PCCNTR.7552865,774790416,CO1.BDOS.7691828,141193702693 RUT ADA FEBRERO 1.pdf,367589,pdf,141193702693 RUT ADA FEBRERO 1.pdf,2026-04-05T00:00:00.000,INSTITUTO DEPARTAMENTAL DE DEPORTES DE ANTIOQUIA,811007127,{'url': 'https://community.secop.gov.co/Public...
248,CO1.PCCNTR.7552865,703566835,CO1.BDOS.7691828,COMPROBANTE EGRESO 153413 VINCULADA A SEGURIDA...,296749,pdf,COMPROBANTE EGRESO 153413 VINCULADA A SEGURIDA...,2025-11-23T00:00:00.000,INSTITUTO DEPARTAMENTAL DE DEPORTES DE ANTIOQUIA,811007127,{'url': 'https://community.secop.gov.co/Public...
249,CO1.PCCNTR.7552865,606117176,CO1.BDOS.7691828,INFORME INDEPORTES 03.pdf,677088,pdf,INFORME INDEPORTES 03.pdf,2025-04-28T00:00:00.000,INSTITUTO DEPARTAMENTAL DE DEPORTES DE ANTIOQUIA,811007127,{'url': 'https://community.secop.gov.co/Public...
250,CO1.PCCNTR.7552865,705076172,CO1.BDOS.7691828,JCC REVISOR AGOSTO.pdf,60268,pdf,JCC REVISOR AGOSTO.pdf,2025-11-25T00:00:00.000,INSTITUTO DEPARTAMENTAL DE DEPORTES DE ANTIOQUIA,811007127,{'url': 'https://community.secop.gov.co/Public...


In [ ]:
%%bash
pip install -q langchain langchain-community faiss-cpu sentence-transformers pypdf requests

### 1. Definir funciones para descargar documentos
Vamos a crear una función que, dado un ID de proceso de compra, busque sus documentos en la API del SECOP, filtre aquellos que tengan URLs de descarga y los almacene localmente en una carpeta.

In [ ]:
import os
import subprocess
import pandas as pd
max=15
def descargar_documentos_desde_dataframe(df_documentos, output_dir="documentos_contrato"):
    os.makedirs(output_dir, exist_ok=True)
    descargados = []

    # Encontrar qué columna tiene las URLs
    columna_url = None
    for col in df_documentos.columns:
        valores_no_nulos = df_documentos[col].dropna()
        if not valores_no_nulos.empty:
            primer_val = valores_no_nulos.iloc[0]
            if isinstance(primer_val, dict) and "url" in primer_val:
                columna_url = col
                break
            elif isinstance(primer_val, str) and (primer_val.startswith("http://") or primer_val.startswith("https://")):
                columna_url = col
                break

    if not columna_url:
        print("No se pudo identificar una columna con URLs válidas.")
        return []

    print(f"Columna con URLs identificada: {columna_url}")

    for idx, row in df_documentos.iterrows():
        url_info = row[columna_url]
        url = url_info["url"] if isinstance(url_info, dict) else url_info

        if isinstance(url, str) and (url.startswith("http") or url.startswith("https")):
            nombre_original = row.get("nombre_archivo", f"documento_{idx}.pdf")
            if not isinstance(nombre_original, str):
                nombre_original = f"documento_{idx}.pdf"

            clean_name = "".join(c for c in nombre_original if c.isalnum() or c in (".", "_", "-")).strip()
            if not clean_name.lower().endswith('.pdf'):
                continue

            filepath = os.path.join(output_dir, clean_name)

            if len(descargados) >= max: # Limitamos a 3 para rapidez
                break

            print(f"Descargando [{len(descargados)+1}]: {clean_name}")
            try:
                # Usamos wget con cabeceras de simulación de navegador que evitan el 403
                cmd = [
                    "wget",
                    "-q",
                    "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
                    "-O", filepath,
                    url
                ]
                resultado = subprocess.run(cmd, capture_output=True, text=True)

                if os.path.exists(filepath) and os.path.getsize(filepath) > 1000:
                    descargados.append(filepath)
                else:
                    print(f"El archivo {clean_name} no pudo descargarse correctamente o está vacío.")
                    if os.path.exists(filepath):
                        os.remove(filepath)
            except Exception as e:
                print(f"Error al descargar {clean_name}: {e}")

    return descargados

### 2. Seleccionar Contrato y Descargar los Documentos
Vamos a seleccionar un contrato del listado de `datiosx` (por ejemplo, el primer registro de la columna `proceso_de_compra`) para descargar su información.

In [ ]:
# Ejecutamos nuevamente la descarga con la lógica robusta de análisis de URLs
archivos_descargados = descargar_documentos_desde_dataframe(datiosdocumentos)
print(f"¡Completado! Se descargaron con éxito {len(archivos_descargados)} documentos en la carpeta 'documentos_contrato'.")

Columna con URLs identificada: url_descarga_documento
Descargando [1/max]: CEDULAREVISORFISCAL.pdf
Descargando [2/max]: 07InformedeSupervisionyReciboaSatisfaccion.pdf
Descargando [3/max]: 05INFORMEDESUPERVICIONYRECIBOASATISFACCION.pdf
Descargando [4/max]: INFORMEDESUPERVISIONYRECIBOASATISFACCION.pdf
Descargando [5/max]: CEDULAREVISORFISCAL.pdf
Descargando [6/max]: INFORMEINDEPORTES01_2026.pdf
Descargando [7/max]: ORDENDEPAGOFRA22886.pdf
Descargando [8/max]: PARAFISCALESADA2..pdf
Descargando [9/max]: COMPROBANTEDEEGRESOFRA.22728.pdf
Descargando [10/max]: INFORMEDESUPERVISIONYRECIBOASATISFACCION.pdf
Descargando [11/max]: INFORMEDESUPERVISIONYRECIBOASATISFACCION.pdf
Descargando [12/max]: COMPROBANTEEGRESO152415VINCULADAAARL.pdf
Descargando [13/max]: TARJETAPROFESIONALREVISORFISCAL.pdf
Descargando [14/max]: COMPROBANTEDEEGRESOFRA.22651.pdf
Descargando [15/max]: INFORMEDESUPERVISIONYRECIBOASATISFACCION.pdf
¡Completado! Se descargaron con éxito 15 documentos en la carpeta 'documentos_contrat

### 3. Procesar los Documentos y Crear el RAG
Usaremos un modelo de embeddings ligero de HuggingFace en GPU para procesar los documentos descargados y guardarlos en un índice de FAISS.

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Cargar solo documentos PDF de la carpeta
loader = PyPDFDirectoryLoader("documentos_contrato")
docs = loader.load()

if len(docs) == 0:
    print("No se pudieron extraer textos de los archivos (puede que no haya PDFs o requieran OCR).")
else:
    # Dividir el texto en fragmentos (Chunks)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
    chunks = text_splitter.split_documents(docs)

    # Inicializar embeddings en GPU (T4)
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        model_kwargs={'device': 'cuda'}
    )

    # Crear la base de datos vectorial con FAISS
    vector_store = FAISS.from_documents(chunks, embeddings)
    print(f"¡Índice RAG creado con éxito! Se procesaron {len(chunks)} fragmentos de texto.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

¡Índice RAG creado con éxito! Se procesaron 148 fragmentos de texto.


### 4. Consultar el RAG usando un LLM ligero o búsqueda semántica
Podemos hacer búsquedas semánticas sobre los documentos del contrato.

In [ ]:
query = "Cuanto debería costar el modelo"

if 'vector_store' in locals():
    resultados = vector_store.similarity_search(query, k=3)
    print(f"Resultados para la consulta: '{query}'\n")
    for i, res in enumerate(resultados):
        print(f"--- Fragmento {i+1} (Origen: {res.metadata.get('source')}) ---")
        print(res.page_content[:400] + "...")
        print("\n")
else:
    print("Asegúrate de que el índice RAG se haya creado correctamente en el paso anterior.")

Resultados para la consulta: 'Cuanto debería costar el modelo'

--- Fragmento 1 (Origen: documentos_contrato/05INFORMEDESUPERVICIONYRECIBOASATISFACCION.pdf) ---
6 $233,927,172 $23.086.776 $137.491.440 $ 96.435.732 58.78% 
7 $233,927,172 $23.086.776 $160.578.216 $ 73.348.956 68.64% 
8 $233,927,172 $24.616.485 $185.194.701 $48.732.471 79.17%
*Información complementaria:  Se pagan 6 horas de desarrollo autorizadas de los ti ckets  48138 - 
48140-48263 
FORMA DE PAGO: INDEPORTES ANTIOQUIA pagará el valor del Contrato mediante pagos parciales 
mensuales de la ...


--- Fragmento 2 (Origen: documentos_contrato/07InformedeSupervisionyReciboaSatisfaccion.pdf) ---
Versión 13
INFORME DE SUPERVISIÓN Y RECIBO A 
SATISFACCIÓN F-CA-58
Aprobación:
12/02/2025
COPIA CONTROLADA
DESEMBOLSO 
305 437 1.45.4599.77.0-
205128.2.3.2.02.02.008.05.  
Servicios $23.086.770
TOTAL A PAGAR $23.086.770
INFORME FINANCIERO GENERAL
N° VALOR DEL CONTRATO 
CON ADICIONES 
EJECUCION  
POR MES  
TOTAL 
EJECUTADO 
PENDIENTE P

### 5. Cargar un Modelo de Lenguaje (LLM) Ligero
Cargaremos el modelo instruccional `Qwen/Qwen2.5-1.5B-Instruct` en la GPU T4 para responder preguntas en español basándonos en el contexto recuperado.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

print("Cargando tokenizador y modelo en la GPU T4...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Crear el pipeline de generación
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.2,
    top_p=0.9,
    do_sample=True
)
print("¡Modelo cargado correctamente en la GPU!")

Cargando tokenizador y modelo en la GPU T4...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

¡Modelo cargado correctamente en la GPU!


### 6. Pipeline de RAG: Rol de Asesor en Contratación Pública
Definiremos la función para estructurar el prompt de contexto, invocar el modelo y mostrar de qué archivo exacto se extrajo la respuesta.

In [ ]:
def responder_consulta_rag(pregunta):
    if 'vector_store' not in locals() and 'vector_store' not in globals():
        return "Error: El índice vectorial FAISS no está listo."

    # 1. Recuperar los fragmentos más relevantes
    resultados_busqueda = vector_store.similarity_search(pregunta, k=3)

    # 2. Construir el bloque de contexto y rastrear las fuentes
    contexto_str = ""
    fuentes = set()
    for idx, doc in enumerate(resultados_busqueda):
        fuente = doc.metadata.get('source', 'Desconocido').split('/')[-1]
        fuentes.add(fuente)
        contexto_str += f"[Fragmento {idx+1} - Fuente: {fuente}]:\n{doc.page_content}\n\n"

    # 3. Prompt estructurado con el rol solicitado
    prompt_sistema = """Eres un Asesor Experto en Contratación Pública en Colombia.
Tu objetivo es responder de manera técnica, objetiva y precisa basándote ÚNICAMENTE en el contexto de documentos proporcionado.
Si la respuesta no se encuentra en el contexto, indícalo amablemente.
Siempre debes citar explícitamente qué documentos o fuentes sustentan tu respuesta en el texto."""

    contenido_mensaje = f"""Contexto de documentos del contrato:
{contexto_str}

Pregunta del usuario: {pregunta} """

    # Aplicar plantilla de chat del modelo
    messages = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": contenido_mensaje}
    ]

    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # 4. Generar respuesta
    outputs = generator(formatted_prompt)
    respuesta = outputs[0]['generated_text'][len(formatted_prompt):]

    # 5. Mostrar resultados
    print("=================== RESPUESTA DEL ASESOR ===================\n")
    print(respuesta.strip())
    print("\n=================== FUENTES CONSULTADAS ===================")
    for f in fuentes:
        print(f"- {f}")

### 7. Probar el Mini RAG con Preguntas Reales

In [ ]:
# Probamos el pipeline completo
responder_consulta_rag(query)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=================== RESPUESTA DEL ASESOR ===================

Para determinar cuánto debería costar el modelo, necesito más información sobre el modelo específico que se está hablando. Sin embargo, puedo decirte cómo se calcula el costo general de los modelos mencionados en los documentos:

El costo total de los modelos se calcula sumando el valor de cada uno de ellos. En este caso, el valor promedio de cada modelo parece ser de aproximadamente $23.086.776.

Por lo tanto, si hay varios modelos, simplemente multiplicaría ese valor por el número de modelos para obtener el costo total. Por ejemplo, si hay 5 modelos, el costo total sería de $23.086.776 * 5 = $115.433.880.

Sin embargo, sin especificar exactamente cuál es el modelo en cuestión, no puedo proporcionar una cifra concreta. Si tienes más detalles sobre el modelo específico, estaré encantado de ayudarte a calcular su costo.

=================== FUENTES CONSULTADAS ===================
- 07InformedeSupervisionyReciboaSatisfaccion.p